# T31 / E14 — Bậc thang kích thước mô hình đọc

**Chạy trong MỘT phiên.** Bản đầu của notebook này bắt chạy hai phiên riêng; không cần thiết, và
đây là lý do.

## Save Version có bị hạ xung không — có, nhưng nó chỉ đụng vào một cột

Hạ xung là chuyện của **phần cứng**, không phải của cách khởi chạy. Save Version chạy trên đúng
card T4 ấy qua papermill, nên nó **không tránh được** hạ xung. Mục 5 `CLAUDE.md` đo được T4 chậm
đi 10–15 % sau vài phút chạy liên tục.

Nhưng phải hỏi tiếp: hạ xung làm hỏng **cái gì**?

| Cột | Hạ xung có đụng tới không |
|---|---|
| macro-F1, F1 từng lớp, ECE | **không** — trọng số chú ý y hệt nhau dù card chạy 1.590 hay 1.200 MHz |
| ms mỗi mẫu | **có** — và đây là một nửa câu hỏi của E14 |

Nên gộp hai cỡ vào một phiên là an toàn cho toàn bộ phần độ chính xác. Chỉ cột chi phí cần xử lý
riêng, và mục 5 `CLAUDE.md` cho sẵn cách: *"chạy mỗi cấu hình một phiên riêng **hoặc đo xen kẽ**"*.

## Đo xen kẽ, tức ô 11

Ô 9 đo chi phí theo thứ tự **7B → 3B → 1.5B → 7B → 3B → 1.5B**, mỗi lượt một tiến trình riêng nạp lại mô
hình từ đầu. Nếu card có trôi trong phiên thì nó trôi lên **cả hai** mô hình như nhau thay vì dồn
hết vào mô hình chạy sau. Hai lượt của cùng một mô hình lệch nhau bao nhiêu chính là thước đo
mức trôi — in ra để đọc, không giấu.

`measure_throughput.py` còn đọc **nhiệt độ GPU** ở mỗi lượt, nên nếu hạ xung xảy ra thì thấy được
trực tiếp chứ không phải suy đoán.

## Chỉ phải chạy HAI nấc, không phải ba

| Nấc | Trích đặc trưng? | Đo chi phí? | Ghi chú |
|---|---|---|---|
| Qwen2.5-7B | **không** | **có** | Độ chính xác lấy từ E02 (0,7451) và E03 (0,7567) |
| Qwen2.5-3B | có | có | |
| Qwen2.5-1.5B | có | có | |

**Vì sao 7B không trích lại nhưng vẫn đo chi phí.** Hai nửa của E14 trả lời câu này khác nhau.
Độ chính xác của 7B đã có, cùng bộ dữ liệu và cùng cách chia đoạn, shard còn nằm sẵn trên máy —
trích lại là đốt 62 phút GPU để dựng lại con số đã có. Nhưng chi phí 528 ms/mẫu của nó đo ở một
**phiên khác** với phiên sắp đo 3B và 1.5B, tức đúng kiểu so sánh mà mục 5 `CLAUDE.md` bảo đừng
làm, và đúng hạn chế mà cột chi phí của E13 đã phải mang. Mười phút đo xen kẽ sửa được chuyện
đó.

## Hai câu E14 trả lời

1. **Câu cũ:** phương pháp chạy được trên phần cứng nhỏ hơn không, mất bao nhiêu điểm — trục chi
   phí của CH2.
2. **Câu mới, do T30 sinh ra:** E13 đo được nhóm chunk-aware **không chuyển** giữa Qwen2.5-7B và
   Sailor2-8B — hai mô hình khác họ huấn luyện. Bậc thang này hỏi nó có chuyển giữa các **cỡ của
   cùng một họ** không. Phép thử nhẹ hơn hẳn: cùng kiến trúc, cùng dữ liệu huấn luyện, chỉ khác
   số tham số.

Vì câu thứ hai, mỗi cỡ có **hai** cấu hình — một chunk-aware và một mốc lookback gộp của chính
nó. T30 dạy bài này bằng một kết luận sai: so chunk-aware của Sailor2 với mốc của Qwen thì chênh
lệch trộn hai biến, không tách được.

## Chi phí ước tính

| Bước | Thời gian |
|---|---|
| Dò kiểu số, hai cỡ | ~10 phút |
| Trích 3B (7.000 mẫu) | ~29 phút |
| Trích 1.5B (7.000 mẫu) | ~18 phút |
| Đo chi phí xen kẽ, sáu lượt | ~22 phút |
| **Tổng** | **~80 phút** cộng thời gian tải mô hình |

**Chấm điểm chạy ở máy cá nhân**, theo quy tắc chốt ở T23.


In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

In [ ]:
# Ô 2 — ba nấc của bậc thang. Không phải sửa gì ở ô này.
#
# Nac 7B co "trich": False. Ly do: E02 va E03 da trich no roi, cung bo du lieu, cung cach chia
# doan, cung nhom dac trung — trich lai la dot 62 phut GPU de dung lai con so da co.
#
# Nhung no VAN nam trong o 9. Chi phi 528 ms/mau cua 7B do o mot PHIEN KHAC voi phien sap do 3B
# va 1.5B, tuc dung kieu so sanh ma muc 5 CLAUDE.md bao dung lam. Muoi phut do xen ke sua duoc
# chuyen do va cho E11 mot duong cong chi phi that su la duong cong.
BAC_THANG = [
    {
        "co": "7B",
        "mo_hinh": "Qwen/Qwen2.5-7B-Instruct",
        "chunk": "configs/e03_chunk_sentence_vihallu.yaml",
        "moc": None,
        "trich": False,
    },
    {
        "co": "3B",
        "mo_hinh": "Qwen/Qwen2.5-3B-Instruct",
        "chunk": "configs/e14_qwen3b_vihallu.yaml",
        "moc": "configs/e14_baseline_lookback_qwen3b.yaml",
        "trich": True,
    },
    {
        "co": "1.5B",
        "mo_hinh": "Qwen/Qwen2.5-1.5B-Instruct",
        "chunk": "configs/e14_qwen15b_vihallu.yaml",
        "moc": "configs/e14_baseline_lookback_qwen15b.yaml",
        "trich": True,
    },
]
CAN_TRICH = [n for n in BAC_THANG if n["trich"]]

print("=" * 78)
for nac in BAC_THANG:
    viec = "trich + do chi phi" if nac["trich"] else "CHI do chi phi (da co dac trung)"
    print(f"  {nac['co']:<6} {nac['mo_hinh']:<30} {viec}")
print("=" * 78)
print(f"  Trich {len(CAN_TRICH)} nac, do chi phi ca {len(BAC_THANG)} nac.")

In [ ]:
# Ô 3 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
!pip install -q --no-deps -e .
!pip install -q bitsandbytes

In [ ]:
# Ô 4 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import load_config
from vihallulens.data.paths import find_raw_dir

problems = []

packages = ("torch", "transformers", "bitsandbytes", "pandas", "accelerate")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
trang_thai = f"THIEU {absent}" if absent else f"du ca {list(packages)}"
print(f"  goi phai co san   : {trang_thai}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("vihallu*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file vihallu tho  : {files or 'KHONG CO'}")
    if not files:
        problems.append("khong thay file vihallu nao trong du lieu tho")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

for nac in BAC_THANG:
    khoas = ("chunk", "moc") if nac["trich"] else ("chunk",)
    for khoa in khoas:
        cfg = load_config(nac[khoa])
        ghi_chu = "<- o 5 se ghi de" if nac["trich"] else "<- da chot tu T07, khong dung toi"
        print(f"  {nac['co']:<6} {khoa:<6}: {Path(nac[khoa]).name:<38} "
              f"exclude_layers {str(cfg.extractor.exclude_layers):<8} {ghi_chu}")

chain = [
    ("o 5", "exclude_layers trong CA BON cau hinh", "compare_dtypes, hai luot"),
    ("o 6", "hai hash moi co phai trung nhau", "kiem"),
    ("o 7/8", "data/processed/vihallu_{split}_<hash>.jsonl", "extract_features"),
    ("o 9", "chi phi ms/mau, do xen ke", "measure_throughput, bon luot"),
]
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<7} {target:<46} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat.")

In [ ]:
# Ô 5 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_attention_hook.py tests/test_drop_nonfinite.py -q

## Dò lớp tràn số — vẫn phải đo cho từng cỡ

Qwen2.5-7B hỏng đúng lớp 27, nhưng hai cỡ này là mô hình khác và T30 cho thấy chuyện này **không
suy ra được**: Sailor2 hỏng theo kiểu hoàn toàn khác — cả mạng hỏng trên 0,7 % mẫu thay vì một
lớp hỏng trên mọi mẫu.

Ô 6 ghi kết quả vào **cả bốn** cấu hình. Ô 7 kiểm hai hash của mỗi cỡ có trùng nhau không — lệch
là mốc lookback sẽ đòi trích lại từ đầu thay vì dùng lại shard.

In [ ]:
# Ô 6 — DÒ LỚP TRÀN SỐ, hai cỡ. Khoảng 10 phút GPU. BẮT BUỘC chạy trước ô 8.
#
# Qwen2.5-7B hong dung lop 27, nhung hai co nay la mo hinh khac, so lop khac. T30 cho thay chuyen
# nay KHONG suy ra duoc: Sailor2 hong theo kieu hoan toan khac Qwen.
#
# Hai mo hinh nay nho hon Sailor2 nhieu nen luot moc bfloat16 se vua bo nho — lan nay co ca phan
# kiem "cac lop con song co bi bop meo khong" ma Sailor2 khong cho duoc.
import ast
import os
import re
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
LOP_TRAN = {}

for nac in CAN_TRICH:
    print("=" * 78)
    print(f"  DO KIEU SO — {nac['co']}  ({nac['mo_hinh']})")
    print("=" * 78)
    proc = subprocess.run(
        ["python", "scripts/compare_dtypes.py", "--model", nac["mo_hinh"],
         "--per-dataset", "10", "--reference", "bfloat16"],
        capture_output=True, text=True, env={**os.environ},
    )
    print(proc.stdout[-7000:])
    if proc.returncode:
        print(proc.stderr[-3000:])
        raise SystemExit(f"do kieu so hong o nac {nac['co']}")

    found = re.search(r"^EXCLUDE_LAYERS=(\[.*\])$", proc.stdout, flags=re.MULTILINE)
    if not found:
        raise SystemExit(f"khong thay dong EXCLUDE_LAYERS= cho nac {nac['co']}")
    bad = sorted(set(ast.literal_eval(found.group(1))))
    LOP_TRAN[nac["co"]] = bad
    print(f"\n  {nac['co']}: lop tran so = {bad if bad else 'KHONG CO'}")

    # Ghi vao CA HAI cau hinh cua co nay. Thieu mot cai thi hai hash lech nhau va moc lookback
    # se doi trich lai tu dau thay vi dung lai shard.
    for khoa in ("chunk", "moc"):
        path = Path(nac[khoa])
        text = path.read_text(encoding="utf-8")
        patched = re.sub(r"^  exclude_layers: \[\]$", f"  exclude_layers: {bad}", text, count=1,
                         flags=re.MULTILINE)
        if patched == text:
            raise SystemExit(f"khong tim thay dong 'exclude_layers: []' trong {nac[khoa]}")
        path.write_text(patched, encoding="utf-8")
        print(f"    da ghi: {nac[khoa]}")

print()
print(f"  Tom tat: {LOP_TRAN}")
print("  NHO commit lai bon file config sau khi chay xong.")

In [ ]:
# Ô 7 — cổng kiểm trước khi tiêu GPU. Vài giây, CPU.
import sys

sys.path.insert(0, "src")
from importlib import reload

import vihallulens.config as config_module

reload(config_module)
HASH = {}
for nac in CAN_TRICH:
    chunk = config_module.load_config(nac["chunk"])
    moc = config_module.load_config(nac["moc"])
    h_chunk = config_module.extraction_hash(chunk)
    h_moc = config_module.extraction_hash(moc)
    HASH[nac["co"]] = h_chunk
    print(f"  {nac['co']:<6} exclude {str(chunk.extractor.exclude_layers):<12} "
          f"hash chunk {h_chunk}  hash moc {h_moc}")
    if not chunk.extractor.exclude_layers and not moc.extractor.exclude_layers:
        print("    (ca hai deu trong — chi dung neu o 6 do duoc that su khong lop nao tran)")
    if h_chunk != h_moc:
        raise SystemExit(
            f"NAC {nac['co']}: HAI HASH KHAC NHAU. Moc lookback se doi trich lai tu dau thay vi "
            f"dung lai shard. Nguyen nhan gan nhu chac chan la o 6 chi ghi duoc vao mot file."
        )

if len(set(HASH.values())) != len(HASH):
    raise SystemExit(f"HAI NAC RA CUNG MOT HASH: {HASH}. Kiem lai model_name trong config.")
print(f"\n  Hai nac, hai hash khac nhau, moi nac hai cau hinh trung hash: {HASH}")

## Trích đặc trưng

**Đọc gì trong lúc chạy:** dòng `lỗi` phải là 0, và dòng `LỚP TRÀN SỐ` nếu xuất hiện thì đọc kỹ —
nó liệt kê lớp nào tràn và bao nhiêu mẫu. Liệt kê *mọi* lớp thì đó là kiểu hỏng của Sailor2, xử
lý bằng bỏ mẫu chứ không bỏ lớp.

In [ ]:
# Ô 8 — trích đặc trưng hai cỡ nhỏ. Khoảng 47 phút. Chạy lại được, có lưu tiến độ.
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

for nac in CAN_TRICH:
    print("=" * 78)
    print(f"  TRICH — {nac['co']}")
    print("=" * 78)
    for split in ("train", "dev", "test"):
        code = os.system(
            f"python scripts/extract_features.py --config {nac['chunk']} --split {split}")
        if code:
            raise SystemExit(f"trich {nac['co']}/{split} hong, ma loi {code}")

## Đo chi phí xen kẽ

Ô này là thứ giữ cho cột `ms/mẫu` của E14 **so được**, dù hai cỡ chạy chung một phiên.

Đọc kết quả bằng cách so **lần 1 với lần 2 của cùng một mô hình**. Hai lần lệch nhau bao nhiêu
chính là mức trôi của card trong phiên. Lệch nhỏ thì cột chi phí dùng được; lệch lớn thì vẫn dùng
được nhưng phải báo cáo mức trôi bên cạnh.

In [ ]:
# Ô 9 — ĐO CHI PHÍ XEN KẼ, ca ba nac. Khoảng 22 phút GPU. Đây là ô giữ cho cột ms/mẫu so được.
#
# Thu tu: 7B -> 3B -> 1.5B -> 7B -> 3B -> 1.5B.
# Moi luot mot tien trinh rieng, nap lai mo hinh tu dau.
#
# Ly do khong do noi nhau tung mo hinh: T4 ha xung 10-15 % sau vai phut chay lien tuc (muc 5
# CLAUDE.md). Do noi nhau thi mo hinh chay sau luon co ve cham hon, va phan cham do bi tinh nham
# thanh khac biet giua hai mo hinh. Xen ke thi phan troi do roi len CA HAI nhu nhau.
#
# Hai luot cua cung mot mo hinh lech nhau bao nhieu CHINH LA thuoc do muc troi — doc no, dung bo
# qua. measure_throughput.py con doc nhiet do GPU moi luot.
import sys

sys.path.insert(0, "src")
from vihallulens.config import load_config

# Hai vong qua ca ba nac: 7B, 3B, 1.5B, roi lap lai. Card troi thi troi len ca ba nhu nhau.
THU_TU = BAC_THANG + BAC_THANG
for vong, nac in enumerate(THU_TU, start=1):
    lan = 1 if vong <= len(BAC_THANG) else 2
    cfg = load_config(nac["chunk"])
    bo_lop = " ".join(str(x) for x in cfg.extractor.exclude_layers)
    ten = f"t31_chiphi_{nac['co'].replace('.', '_')}_lan{lan}"
    print("=" * 78)
    print(f"  LUOT {vong}/{len(THU_TU)} — {nac['co']}, lan {lan}   run_name {ten}")
    print("=" * 78)
    lenh = (f"python scripts/measure_throughput.py --model {nac['mo_hinh']} "
            f"--per-tier 8 --run-name {ten}")
    if bo_lop:
        lenh += f" --exclude-layers {bo_lop}"
    code = os.system(lenh)
    if code:
        raise SystemExit(f"do chi phi hong o luot {vong}, ma loi {code}")

print()
print("  Doc gi: so sanh lan 1 voi lan 2 CUA CUNG MOT MO HINH.")
print("  Lech nho  -> card khong troi dang ke, cot ms/mau so duoc.")
print("  Lech lon  -> card co troi, phai bao cao muc troi ben canh con so chi phi.")

In [ ]:
# Ô 10 — soi shard hai cỡ vừa trích. Vài giây, CPU. KHÔNG dừng notebook.
#
# Khong raise du shard co nan: mot shard hong la thu CAN dem ve nhat de chan doan. Bai hoc T30.
for nac in CAN_TRICH:
    print("=" * 78)
    print(f"  SOI SHARD — {nac['co']}")
    print("=" * 78)
    proc = subprocess.run(["python", "scripts/inspect_shard.py", "--config", nac["chunk"]],
                          capture_output=True, text=True)
    print(proc.stdout[-6000:])
    if proc.returncode:
        print(proc.stderr[-2000:])

print()
print("  DU SHARD CO NAN HAY KHONG, VAN CHAY O 11 DE MANG VE.")

## Chấm điểm — KHÔNG chạy ở đây

Quy tắc chốt ở T23: mọi phép so sánh phải chấm trên **cùng một máy**, vì điểm dev lệch tới 0,0075
giữa Kaggle và máy cá nhân do bộ giải tối ưu hội tụ khác nhau.

In [ ]:
# Ô 11 — lấy kết quả về. Vài giây.
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

out = Path("/kaggle/working/ket_qua_t31")
out.mkdir(exist_ok=True)
for nac in CAN_TRICH:
    run = extraction_hash(load_config(nac["chunk"]))
    for split in ("train", "dev", "test"):
        src = Path(f"data/processed/vihallu_{split}_{run}.jsonl")
        if src.exists():
            shutil.copy(src, out / src.name)
    for khoa in ("chunk", "moc"):
        shutil.copy(nac[khoa], out / Path(nac[khoa]).name)
if Path("results/runs.jsonl").exists():
    shutil.copy("results/runs.jsonl", out / "runs_t31.jsonl")

for f in sorted(out.iterdir()):
    print(f"  {f.name:<44} {f.stat().st_size / 1e6:>8.1f} MB")

print("""
Tai het thu muc ket_qua_t31 ve may, dat vao:
  *.jsonl (vihallu_*)  ->  data/processed/
  *.yaml               ->  configs/   (GHI DE — chung mang exclude_layers da do)
  runs_t31.jsonl       ->  giu lai, no chua bon luot do chi phi cua o 9

Roi bao lai de cham diem o may ca nhan, 0 giay GPU.
""")